# Day 4 — Prediction Engine & Model Benchmarking

Architecture **Layer 4**: learn from historical campaigns to predict **funding
success**, and derive honest venture scores. This notebook walks the Day-4
artifacts (built by `src/models/`); see `docs/day4_roadmap.md`.

Flow: split → LogReg baseline (Week 6) → benchmark → tune → calibrate →
venture scores → prediction engine → **test scored once**.

In [1]:
import sys, os, json, warnings, logging
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir(os.path.abspath(".."))
sys.path.insert(0, os.getcwd())
warnings.filterwarnings("ignore"); logging.disable(logging.WARNING)
import numpy as np, pandas as pd

def load(p):
    return json.load(open(p))

## Phase A/B — Problem & data split

Binary classification: P(funding success) from pre-launch features. Base rate
40.4% → majority baseline (No-Information Rate) = 59.6% accuracy. Validation is
carved from train; the **test set is the untouched Day-3 holdout**.

In [2]:
from src.models.data_split import ModelDataSplitter
data = ModelDataSplitter().load()
pd.DataFrame(data.summary()).T

,n,pos_rate
fit,212136.0,0.4038
val,53035.0,0.4038
test,66293.0,0.4038
n_features,865.0,865.0000


## Phase C — Logistic Regression baseline (mandatory Week 6)

L2 / L1 / ElasticNet vs a Dummy floor. All regularizers tie (n/p ≈ 245).

In [3]:
lr = load("artifacts/models/logreg_results.json")
rows = {k: v["metrics"] for k, v in lr.items() if k != "_context"}
pd.DataFrame(rows).T[["accuracy", "f1", "roc_auc", "pr_auc"]]

,accuracy,f1,roc_auc,pr_auc
logreg_l2,0.6768,0.5486,0.7269,0.6264
logreg_l1,0.6769,0.5486,0.7269,0.6264
logreg_elasticnet,0.6765,0.5482,0.7269,0.6264
dummy_majority,0.5962,0.0000,0.5000,0.4038


## Phase D — Benchmark (boosting > bagging > linear)

In [4]:
b = load("artifacts/evaluations/benchmark.json")
rows = {k: {**v["metrics"], "train_s": v["train_time_s"]}
        for k, v in b.items() if k != "_context"}
pd.DataFrame(rows).T.sort_values("pr_auc", ascending=False)[
    ["pr_auc", "roc_auc", "f1", "recall", "train_s"]]

,pr_auc,roc_auc,f1,recall,train_s
hist_gradient_boosting,0.6805,0.7703,0.6077,0.5627,30.2
lightgbm,0.6798,0.7697,0.6071,0.5614,2.9
xgboost,0.6754,0.7675,0.6022,0.5538,3.8
random_forest,0.6643,0.7592,0.5841,0.5191,115.4
extra_trees,0.6589,0.7535,0.5803,0.5168,186.9


## Phase E/F — Tuned + calibrated LightGBM

RandomizedSearchCV on PR-AUC; then Platt/Isotonic calibration + threshold choice.

In [5]:
cfg = load("artifacts/models/best_config.json")
cal = load("artifacts/evaluations/calibration.json")
print("tuned best CV PR-AUC:", cfg["best_cv_score"])
print("best calibration:", cal["best_method"], "| Brier:",
      cal["results"][cal["best_method"]]["brier"])
print("F1-optimal threshold:", cal["threshold"]["f1_optimal"],
      "| recall", cal["threshold"]["recall_at_0.5"], "->", cal["threshold"]["recall_at_optimal"])
pd.Series(cfg["best_params"]).to_frame("best_params")

tuned best CV PR-AUC: 0.6872
best calibration: sigmoid | Brier: 0.1872
F1-optimal threshold: 0.329 | recall 0.5762 -> 0.817


,best_params
subsample,0.70
scale_pos_weight,1.00
reg_lambda,1.00
reg_alpha,0.10
num_leaves,127.00
n_estimators,300.00
min_child_samples,50.00
max_depth,-1.00
learning_rate,0.05
colsample_bytree,0.80


## Phase G/H — Prediction engine on a raw campaign

Raw campaign → Day-3 pipeline → calibrated model → threshold → venture scores.
Deferred scores (founder/market/...) return `None` — never fabricated.

In [6]:
from src.models.prediction_engine import PredictionEngine
raw = pd.read_csv("data/processed/kickstarter_processed.csv").drop(
    columns=["target", "state"]).sample(3, random_state=0)
engine = PredictionEngine()
for o in engine.predict(raw):
    print(f"p={o.funding_probability:.3f} -> {o.predicted_label:11s} "
          f"startup_score={o.scores['startup_score']['value']:>3} "
          f"readiness={o.scores['investment_readiness']['value']}")
print("\ndeferred (never fabricated):",
      {k: engine.predict(raw)[0].scores[k]['value']
       for k in ['founder_strength', 'market_fit', 'growth_potential']})

p=0.333 -> successful  startup_score= 33 readiness=Low
p=0.820 -> successful  startup_score= 82 readiness=High
p=0.219 -> failed      startup_score= 22 readiness=Low

deferred (never fabricated): {'founder_strength': None, 'market_fit': None, 'growth_potential': None}


## Phase J — Final evaluation on the untouched TEST set (scored once)

The honest generalization number. Validation ≈ test PR-AUC ⇒ model selection did
not overfit.

In [7]:
from src.models.evaluation import ModelEvaluator
report = ModelEvaluator().run(data.X_test, data.y_test)
print("threshold:", report["threshold"], "| n_test:", report["n_test"])
print("NIR accuracy floor 0.5962 | PR-AUC base rate 0.4038")
pd.Series(report["test_metrics"]).to_frame("test")

threshold: 0.329 | n_test: 66293
NIR accuracy floor 0.5962 | PR-AUC base rate 0.4038


,test
accuracy,0.6713
precision,0.5646
recall,0.8131
f1,0.6664
roc_auc,0.7763
pr_auc,0.6883
brier,0.1873


In [8]:
cm = report["confusion_matrix"]
pd.DataFrame([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]],
             index=["actual failed", "actual successful"],
             columns=["pred failed", "pred successful"])

,pred failed,pred successful
actual failed,22735,16788
actual successful,5004,21766


## Summary

| stage | result |
|---|---|
| Baseline (LogReg L2) | PR-AUC 0.626 |
| Best model (tuned+calibrated LightGBM) | test PR-AUC ~0.69 |
| vs No-Information Rate | +0.075 accuracy, +0.28 PR-AUC |
| Calibration | Brier ~0.187 (already well-calibrated) |
| Recall (threshold 0.33) | ~0.81 |
| Generalization | validation ≈ test (no overfit) |

Deliverable: calibrated `PredictionEngine` (Layer 4) + serialized bundle. Plots
in `artifacts/evaluations/`; full report in `docs/day4_model_report.md`.
**Next (Day 5):** explainability — SHAP + coefficient analysis (Week 7).